# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [2]:
 !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [3]:
 !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [4]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [5]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [6]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [7]:
doc_path =  "./state_of_the_union.txt"

In [8]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [9]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [10]:
from langchain_community.document_loaders import WebBaseLoader

In [11]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [12]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [13]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [14]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [15]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/tmp/ipykernel_20642/1834837320.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [16]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

 I should use the State of Union QA System to answer this question.
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the state of the union address?"
Observation:  Biden said that he nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.
Thought: I now know the final answer.
Final Answer: Biden nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "Biden nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}

In [17]:
agent.invoke("Why use ruff over flake8?")

 Ruff is a python linter that has some advantages over flake8, so it's worth considering.
Action: Ruff QA System
Action Input: "What are the advantages of using ruff over flake8?"
Observation:  Ruff has a larger rule set, supports automatic fixing of lint violations, and does not require the installation of additional plugins. It also has better compatibility with Black and can be used as a formatter as well as a linter.
Thought: These advantages make ruff a more comprehensive and user-friendly tool compared to flake8.
Action: State of Union QA System
Action Input: "What are the current features of ruff?"
Observation:  I don't know.
Thought: I should try asking a more specific question to get a better answer.
Action: Ruff QA System
Action Input: "What are the main features of ruff?"
Observation:  The main features of Ruff include compatibility with Black and Flake8, support for Python 3.7 and above, and the ability to be used as a linter or formatter independently. It also has the abil

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff has a larger rule set, supports automatic fixing of lint violations, and does not require the installation of additional plugins. It also has better compatibility with Black and can be used as a formatter as well as a linter. Additionally, it has features such as compatibility with other popular tools and the ability to be used as a linter or formatter independently.'}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [18]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [19]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [20]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

 I should use the State of Union QA System to answer this question.
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the state of the union address?"
Observation:  Biden said that he nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': " Biden said that he nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}

In [21]:
agent.invoke("Why use ruff over flake8?")

 Ruff is a python linter that has some unique features compared to flake8, so it may be useful in certain situations.
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation:  Ruff offers a larger rule set and better compatibility with other tools like Black. It also has the ability to automatically fix its own lint violations. However, it does not support custom lint rules like Flake8 does.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': ' Ruff offers a larger rule set and better compatibility with other tools like Black. It also has the ability to automatically fix its own lint violations. However, it does not support custom lint rules like Flake8 does.'}

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [22]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [23]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [24]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

 I should check if the president mentioned any tools related to Jupyter Notebooks in the state of the union address.
Action: State of Union QA System
Action Input: "What tools were mentioned in the state of the union address related to Jupyter Notebooks?"
Observation:  None.
Thought: I should check if ruff uses any tools to run over Jupyter Notebooks.
Action: Ruff QA System
Action Input: "What tools does ruff use to run over Jupyter Notebooks?"
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought: I now know the final answer.
Final Answer: Ruff uses nbQA to run over Jupyter Notebooks. The president did not mention this tool in the state of the union address.

> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks. The president did not mention this tool in the state of the union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own!

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:**
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



In [25]:
!wget -O pride_and_prejudice.txt https://www.gutenberg.org/files/1342/1342-0.txt
from langchain_community.document_loaders import TextLoader

loader = TextLoader("pride_and_prejudice.txt", encoding="utf-8")
docs = loader.load()

--2026-08-04 18:37:54--  https://www.gutenberg.org/files/1342/1342-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738046 (721K) [text/plain]
Saving to: ‘pride_and_prejudice.txt’

pride_and_prejudice 100%[===================>] 720.75K   538KB/s    in 1.3s    

2026-08-04 18:38:02 (538 KB/s) - ‘pride_and_prejudice.txt’ saved [738046/738046]



In [26]:
print(len(docs))
print(docs[0].page_content[:1000])

1
*** START OF THE PROJECT GUTENBERG EBOOK 1342 ***




                            [Illustration:

                             GEORGE ALLEN
                               PUBLISHER

                        156 CHARING CROSS ROAD
                                LONDON

                             RUSKIN HOUSE
                                   ]

                            [Illustration:

               _Reading Jane’s Letters._      _Chap 34._
                                   ]




                                PRIDE.
                                  and
                               PREJUDICE

                                  by
                             Jane Austen,

                           with a Preface by
                           George Saintsbury
                                  and
                           Illustrations by
                             Hugh Thomson

                         [Illustration: 1894]

                       Ruskin       156. Chari

In [27]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(docs)

print(f"Created {len(texts)} chunks")

Created 821 chunks


In [28]:
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="pride_and_prejudice_collection")

print("Pride and Prejudice vector store created.")

Pride and Prejudice vector store created.


In [29]:
pride_and_prejudice = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [30]:
tools_new = [
    Tool(
        name="Pride and Prejudice QA System",
        func=pride_and_prejudice.run,
        description="useful for when you need to answer questions about Pride and Prejudice novel . Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [31]:
agent_new = initialize_agent(
    tools_new, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [32]:
agent_new.invoke(
    "Why does Elizabeth initially dislike Mr. Darcy?"
)

 Elizabeth's initial dislike of Mr. Darcy is a key plot point in Pride and Prejudice, so it must be important to understand why.
Action: Pride and Prejudice QA System
Action Input: "Why does Elizabeth initially dislike Mr. Darcy?"
Observation:  Elizabeth initially dislikes Mr. Darcy because of his pride and snobbish behavior towards her and her family. She also believes that he is responsible for separating her sister Jane from Mr. Bingley.
Thought: This makes sense, but I wonder if there are any other reasons for her dislike.
Action: Pride and Prejudice QA System
Action Input: "Are there any other reasons for Elizabeth's initial dislike of Mr. Darcy?"
Observation:  Yes, there are other reasons for Elizabeth's initial dislike of Mr. Darcy. These include his pride and arrogance, his initial snubbing of her at the Meryton ball, and his involvement in separating Jane and Bingley. Additionally, Elizabeth's prejudice against him is fueled by the negative opinions of others, such as Wickham 

{'input': 'Why does Elizabeth initially dislike Mr. Darcy?',
 'output': 'Elizabeth initially dislikes Mr. Darcy because of his pride and snobbish behavior, his involvement in separating Jane and Bingley, and her own prejudices influenced by the opinions of others. However, as she gets to know him better and learns more about his true character, her feelings towards him evolve and she eventually falls in love with him.'}

In [33]:
agent_new.invoke(
    "How many sisters does Elizabeth Bennet have and what are their names?"
)

 I should use the Pride and Prejudice QA System to answer this question.
Action: Pride and Prejudice QA System
Action Input: "How many sisters does Elizabeth Bennet have and what are their names?"
Observation:  Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.
Thought: I should use the Ruff QA System to answer this question.
Action: Ruff QA System
Action Input: "How many sisters does Elizabeth Bennet have and what are their names?"
Observation:  Elizabeth Bennet has four sisters: Jane, Mary, Catherine, and Lydia.
Thought: I now know the final answer.
Final Answer: Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.

> Finished chain.


{'input': 'How many sisters does Elizabeth Bennet have and what are their names?',
 'output': 'Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.'}

In [34]:
agent_new.invoke(
    "How many sisters does Elizabeth Bennet have and what are their names? Then write a small Python class representing them and explain how Ruff would check the code."
)

 I should first find the answer to the first question and then think about how Ruff would check the code.
Action: Pride and Prejudice QA System
Action Input: How many sisters does Elizabeth Bennet have and what are their names?
Observation:  Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.
Thought: Now I should think about how Ruff would check the code.
Action: Ruff QA System
Action Input: A small Python class representing the sisters
Observation:  I don't know.
Thought: I should try to write a small Python class representing the sisters and see how Ruff would check it.
Action: Ruff QA System
Action Input: A small Python class representing the sisters
Observation:  I don't know.
Thought: I should try to write a small Python class representing the sisters and see how Ruff would check it.
Action: Ruff QA System
Action Input: A small Python class representing the sisters
Observation:  I don't know.
Thought: I should try to write a small Python class representing the sister

{'input': 'How many sisters does Elizabeth Bennet have and what are their names? Then write a small Python class representing them and explain how Ruff would check the code.',
 'output': "I don't know."}

In [35]:

tools_new = [
    Tool(
        name="Pride and Prejudice QA System",
        func=pride_and_prejudice.run,
        description="useful for when you need to answer questions about Pride and Prejudice novel . Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [36]:
agent_new = initialize_agent(
    tools_new, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [37]:
agent_new.invoke(
    "Why does Elizabeth initially dislike Mr. Darcy?"
)

 Elizabeth's initial dislike of Mr. Darcy is a key plot point in Pride and Prejudice, so it must be important to understand why.
Action: Pride and Prejudice QA System
Action Input: "Why does Elizabeth initially dislike Mr. Darcy?"
Observation:  Elizabeth initially dislikes Mr. Darcy because of his pride and snobbish behavior towards her and her family. She also believes that he is responsible for separating her sister Jane from Mr. Bingley.


> Finished chain.


{'input': 'Why does Elizabeth initially dislike Mr. Darcy?',
 'output': ' Elizabeth initially dislikes Mr. Darcy because of his pride and snobbish behavior towards her and her family. She also believes that he is responsible for separating her sister Jane from Mr. Bingley.'}

In [38]:
agent_new.invoke(
    "How many sisters does Elizabeth Bennet have and what are their names?"
)

 I should use the Pride and Prejudice QA System to answer this question.
Action: Pride and Prejudice QA System
Action Input: "How many sisters does Elizabeth Bennet have and what are their names?"
Observation:  Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.


> Finished chain.


{'input': 'How many sisters does Elizabeth Bennet have and what are their names?',
 'output': ' Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.'}

In [39]:
agent_new.invoke(
    "How many sisters does Elizabeth Bennet have and what are their names? Then write a small Python class representing them and explain how Ruff would check the code."
)

 I should first find the answer to the first question and then think about how Ruff would check the code.
Action: Pride and Prejudice QA System
Action Input: How many sisters does Elizabeth Bennet have and what are their names?
Observation:  Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.


> Finished chain.


{'input': 'How many sisters does Elizabeth Bennet have and what are their names? Then write a small Python class representing them and explain how Ruff would check the code.',
 'output': ' Elizabeth Bennet has four sisters: Jane, Mary, Kitty, and Lydia.'}

Reflection: The agent picked better the right tool(s) when we set return_direct=True.